# 03 · Build a clean Silver view
The date window comes from widgets. Run this notebook after Bronze; rerunning replaces the view with the selected window. Zero-distance rides are retained and flagged.

In [ ]:
from quality_rules import month_window
import re

dbutils.widgets.text("start_month", "2026-03", "First month (YYYY-MM)")
dbutils.widgets.text("end_month", "2026-05", "Last month (YYYY-MM, inclusive)")
dbutils.widgets.text("catalog", "nyc_mobility", "Target catalog")
dbutils.widgets.text("bronze_schema", "nyc_bronze", "Bronze schema")
dbutils.widgets.text("silver_schema", "nyc_silver", "Silver schema")

start, end_exclusive = month_window(dbutils.widgets.get("start_month"), dbutils.widgets.get("end_month"))
catalog = dbutils.widgets.get("catalog").strip()
bronze_schema = dbutils.widgets.get("bronze_schema").strip()
silver_schema = dbutils.widgets.get("silver_schema").strip()
for identifier in (catalog, bronze_schema, silver_schema):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", identifier):
        raise ValueError(f"Invalid catalog or schema: {identifier!r}")

bronze_table = f"{catalog}.{bronze_schema}.green_taxi_raw"
silver_view = f"{catalog}.{silver_schema}.vw_green_taxi_clean"
print(f"Silver pickup window: {start} through {end_exclusive} (end exclusive)")

In [ ]:
# This is a view: no duplicate Silver rows are written on a rerun.
spark.sql(f"""
    CREATE OR REPLACE VIEW {silver_view} AS
    SELECT
        lpep_pickup_datetime AS pickup_datetime,
        lpep_dropoff_datetime AS dropoff_datetime,
        PULocationID AS pickup_zone_id,
        DOLocationID AS dropoff_zone_id,
        passenger_count,
        trip_distance,
        fare_amount,
        tip_amount,
        total_amount,
        trip_distance = 0 AS is_zero_distance
    FROM {bronze_table}
    WHERE lpep_pickup_datetime >= TIMESTAMP_NTZ '{start} 00:00:00'
      AND lpep_pickup_datetime < TIMESTAMP_NTZ '{end_exclusive} 00:00:00'
      AND lpep_dropoff_datetime >= lpep_pickup_datetime
""")
display(spark.sql(f"""
    SELECT COUNT(*) AS silver_rows,
           COUNT_IF(is_zero_distance) AS zero_distance_rows
    FROM {silver_view}
"""))